# 07 · LightGBM baseline

Untuned LightGBM on the same v1 windows. Primary candidate family before Optuna.

In [1]:
import pandas as pd

from cross_model_drift.data import load_split
from cross_model_drift.features import target_vector
from cross_model_drift.metrics import quality_metrics
from cross_model_drift.models import train_lightgbm
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.tracking import clearml_task, log_metrics, log_parameters

nb = setup_model_session()
train = load_split("v1", "train", nb.config, engine=nb.engine)
valid = load_split("v1", "validation", nb.config, engine=nb.engine)
test = load_split("v1", "test", nb.config, engine=nb.engine)

In [2]:
model = train_lightgbm(
    train,
    target_vector(train),
    valid,
    target_vector(valid),
    threshold=nb.threshold,
)
valid_metrics = quality_metrics(target_vector(valid), model.predict_proba(valid), threshold=nb.threshold)
test_metrics = quality_metrics(target_vector(test), model.predict_proba(test), threshold=nb.threshold)
pd.DataFrame([valid_metrics, test_metrics], index=["validation", "test"])

[50]	train's average_precision: 0.463452	valid's average_precision: 0.3724
[100]	train's average_precision: 0.470192	valid's average_precision: 0.376285
[150]	train's average_precision: 0.47466	valid's average_precision: 0.377576
[200]	train's average_precision: 0.478482	valid's average_precision: 0.377828
[250]	train's average_precision: 0.481645	valid's average_precision: 0.378019
[300]	train's average_precision: 0.484421	valid's average_precision: 0.378085


,precision,recall,f1,pr_auc,roc_auc,threshold,n,n_positive,positive_rate
validation,0.686334,0.324865,0.440993,0.378181,0.851222,0.5,546677,11100,0.020304
test,0.694614,0.332303,0.449544,0.394836,0.852214,0.5,534680,11643,0.021776


In [3]:
path = model.save(nb.artifacts / "models" / "v1_lightgbm.joblib")
with clearml_task("train_lightgbm", config=nb.config, task_type="training", tags=["v1", "lightgbm"], init=True) as task:
    log_parameters(task, model.params)
    log_metrics(task, test_metrics, title="v1_test")
path

ClearML Task: created new task id=8934aa0e45e44afa8f7567015ce53cb4
ClearML results page: http://localhost:8080/projects/23d7eaf499d345e2b9df79587a66a66c/tasks/8934aa0e45e44afa8f7567015ce53cb4/output/log
2026-08-23 19:22:26,391 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


PosixPath('/Users/mitter/aventures/code/cross-model-drift/artifacts/models/v1_lightgbm.joblib')